# Harry Potter RAG — Lab 3: Evaluation

Continuing from [Lab 2: Evaluation Prep](./lab2-evaluation-prep.ipynb), which produced
the labelled set at `labs/eval/qa_candidates.jsonl`.

This lab measures the pipeline from Lab 1: it loads the labelled questions, runs the
retrieval and generation stages, and puts numbers behind every design choice.

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Goal of Lab 3: RAG Evaluation & Ablation Matrix

We take the baseline pipeline and measure how different design choices actually move
the numbers. Instead of treating RAG like a black box, we split evaluation into two
separate questions:

1. Did we retrieve the right context?
2. Did the LLM generate a correct, grounded answer?

### What we're testing

1. **Ground-truth evaluation** — load the labelled Q&A set: questions with exact
   evidence snippets, multi-chapter questions, and out-of-scope queries that test
   refusal behaviour.

2. **Ablation grid** — an automated matrix comparison across the main knobs:
    - **Chunking:** fixed-size (256 vs. 512 tokens) vs. chapter-aware boundaries.
    - **Retrieval:** dense (Titan V2 embeddings), sparse (BM25) and hybrid (Reciprocal Rank Fusion).

3. **Separating the failures**
    - **Retriever metrics:** `Hit@k`, `Recall@k` and `MRR` — did the labelled evidence make the context window?
    - **Generator metrics:** an LLM-as-a-Judge for answer correctness and faithfulness, plus false-abstention rate.

4. **Results leaderboard** — every run logs to `eval/results.csv`, so architectural
   choices are backed by numbers rather than vibes.

## Preconditions

- **Lab 1 pipeline** — embeddings, the Chroma collection, BM25, and the `ask()` loop.
- **Lab 2 output** — `labs/eval/qa_candidates.jsonl`. Filter on `locatable` and treat
  `evidence_chunk_ids` as the retrieval label. Lab 3 never regenerates it.

## Build order

1. Load the labelled set and the chunk index.
2. Build the config matrix: chunk size x chunking strategy x retriever.
3. Retrieve, then score `Hit@k` / `Recall@k` / `MRR` against `evidence_chunk_ids`.
4. Score generation: `EM` / `F1`, plus a judge for correctness and faithfulness.
5. Emit one row per question to `labs/eval/results.csv`.

## Import dependencies

In [1]:
from pathlib import Path
import json
import collections

## Load the evaluation set

In [2]:
# Labelled evaluation set produced by Lab 2 (Evaluation Prep).

def repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p


EVAL = repo_root() / "labs" / "eval"
rows = [json.loads(line) for line in open(EVAL / "qa_candidates.jsonl", encoding="utf-8") if line.strip()]
usable = [r for r in rows if r.get("locatable")]

print(f"{len(rows)} candidates | {len(usable)} usable (locatable)")
print("types:", dict(collections.Counter(r["type"] for r in usable)))
print("chapters:", len({r["chapter"] for r in usable}))
usable[0]

69 candidates | 65 usable (locatable)
types: {'factoid': 62, 'multi-hop': 3}
chapters: 17


{'id': 'cand-hp1-01-01',
 'book': 1,
 'chapter': 1,
 'chapter_name': 'Chapter 1 - The Boy Who Lived',
 'type': 'factoid',
 'question': 'What is the name of the firm where Mr. Dursley works?',
 'answer': 'Grunnings',
 'evidence': 'Mr. Dursley was the director of a firm called Grunnings, which made drills.',
 'evidence_chunk_ids': ['hp1-ch01-0001'],
 'n_spans': 1,
 'locatable': True,
 'distinctive': True,
 'unmatched_spans': [],
 'status': 'draft'}